# Ray Serve Router Benchmark — Evaluation

Loads `results/<run-id>/guidellm.json` for routers (`p2c`, `round_robin`, `consistent_hash`, `prefix`),
builds a client-side comparison table (TTFT / E2E / RPS / errors), and plots primary metrics.

When present, also loads `results/<run-id>/assembled_chat_completions.json` (Proxy tee of sim
responses) for **pod distribution** and **`cached_tokens`** stats — evidence of prefix-cache affinity
that GuideLLM does not record (`cached_tokens: null` in `guidellm.json`).

**Fair comparison:** compare runs that used the same dataset and sim latency stack
(all `prefix_shared` + per-token sims from `docker-compose.yml`, cold KV via
`run_router_bench.ps1`). Do not mix runs from different prompt sets or latency configs.

In [ ]:


import json
import warnings
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots

RESULTS_DIR = Path("results")
ROUTERS = ["p2c", "round_robin", "consistent_hash", "prefix"]
PLOTS_DIR = RESULTS_DIR / "plots"
SAVE_PLOTS = False  # set True to also write results/plots/*.html
ASSEMBLED_NAME = "assembled_chat_completions.json"


def _dist(metrics: dict, key: str, status: str = "successful") -> dict:
    """Safe access into a StatusDistributionSummary: metrics[key][status]."""
    return (metrics.get(key) or {}).get(status) or {}


def _requests_made(benchmark: dict) -> dict:
    """successful / errored / incomplete / total request counts."""
    made = (benchmark.get("run_stats") or {}).get("requests_made")
    if made:
        return made
    made = (benchmark.get("scheduler_metrics") or {}).get("requests_made")
    if made:
        return made
    made = benchmark.get("request_totals")
    if made:
        return made
    metrics = benchmark.get("metrics") or {}
    totals = metrics.get("request_totals")
    if totals:
        return totals
    counts = {
        status: _dist(metrics, "requests_per_second", status).get("count", 0)
        for status in ("successful", "errored", "incomplete")
    }
    counts["total"] = sum(counts.values())
    return counts


def load_summary(router: str) -> dict | None:
    """Load GuideLLM summary for one run-id folder, or None if missing."""
    path = RESULTS_DIR / router / "guidellm.json"
    if not path.is_file():
        warnings.warn(f"skip {router}: missing {path}")
        return None
    data = json.loads(path.read_text(encoding="utf-8"))
    benchmark = data["benchmarks"][-1]
    metrics = benchmark.get("metrics") or {}
    made = _requests_made(benchmark)
    total = made.get("total") or sum(
        made.get(s, 0) for s in ("successful", "errored", "incomplete")
    )
    error_rate = (
        ((made.get("errored", 0) + made.get("incomplete", 0)) / total)
        if total
        else float("nan")
    )

    def stat(key: str, stat_name: str) -> float:
        d = _dist(metrics, key)
        if stat_name in d:
            return d.get(stat_name, float("nan"))
        return (d.get("percentiles") or {}).get(stat_name, float("nan"))

    return {
        "router": router,
        "requests_total": total,
        "requests_successful": made.get("successful", 0),
        "requests_errored": made.get("errored", 0),
        "requests_incomplete": made.get("incomplete", 0),
        "error_rate": error_rate,
        "ttft_mean_ms": stat("time_to_first_token_ms", "mean"),
        "ttft_p50_ms": stat("time_to_first_token_ms", "p50"),
        "ttft_p90_ms": stat("time_to_first_token_ms", "p90"),
        "ttft_p99_ms": stat("time_to_first_token_ms", "p99"),
        "e2e_mean_s": stat("request_latency", "mean"),
        "e2e_p50_s": stat("request_latency", "p50"),
        "e2e_p90_s": stat("request_latency", "p90"),
        "e2e_p99_s": stat("request_latency", "p99"),
        "throughput_rps": stat("requests_per_second", "mean"),
        "throughput_tokens_s": stat("tokens_per_second", "mean"),
        "tpot_mean_ms": stat("time_per_output_token_ms", "mean"),
        "itl_mean_ms": stat("inter_token_latency_ms", "mean"),
    }


rows = [r for r in (load_summary(name) for name in ROUTERS) if r is not None]
if not rows:
    raise FileNotFoundError(f"no guidellm.json under {RESULTS_DIR} for {ROUTERS}")
summary = pd.DataFrame(rows).set_index("router")
display(summary)

plot_specs = [
    ("ttft_mean_ms", "ttft_p99_ms", "TTFT (ms)"),
    ("e2e_mean_s", "e2e_p99_s", "E2E latency (s)"),
    ("throughput_rps", None, "Throughput (req/s)"),
    ("throughput_tokens_s", None, "Throughput (tokens/s)"),
    ("error_rate", None, "Error + incomplete rate"),
    ("tpot_mean_ms", "itl_mean_ms", "TPOT vs ITL (ms)"),
]
fig = make_subplots(
    rows=2,
    cols=3,
    subplot_titles=[title for _, _, title in plot_specs],
)
routers = list(summary.index)
for i, (col_a, col_b, _title) in enumerate(plot_specs):
    row, col = divmod(i, 3)
    row, col = row + 1, col + 1
    if col_b is None:
        fig.add_trace(
            go.Bar(name=col_a, x=routers, y=summary[col_a], showlegend=False),
            row=row,
            col=col,
        )
    else:
        fig.add_trace(
            go.Bar(name=col_a, x=routers, y=summary[col_a], legendgroup=col_a),
            row=row,
            col=col,
        )
        fig.add_trace(
            go.Bar(name=col_b, x=routers, y=summary[col_b], legendgroup=col_b),
            row=row,
            col=col,
        )
fig.update_layout(
    height=700,
    width=1100,
    
    barmode="group",
    title_text="GuideLLM router comparison",
    legend_title_text="metric",
)
fig.show()

if SAVE_PLOTS:
    PLOTS_DIR.mkdir(parents=True, exist_ok=True)
    fig.write_html(PLOTS_DIR / "router_comparison.html")

,requests_total,requests_successful,requests_errored,requests_incomplete,error_rate,ttft_mean_ms,ttft_p50_ms,ttft_p90_ms,ttft_p99_ms,e2e_mean_s,e2e_p50_s,e2e_p90_s,e2e_p99_s,throughput_rps,throughput_tokens_s,tpot_mean_ms,itl_mean_ms
router,,,,,,,,,,,,,,,,,
p2c,500,500,0,0,0.0,122.953440,120.510578,128.900290,163.635969,0.972535,0.966720,1.483912,1.826168,2.988575,2226.912594,22.663749,20.258605
round_robin,500,500,0,0,0.0,52.647227,34.587622,164.849043,172.464609,0.876295,0.865747,1.373331,1.802293,2.991660,2225.633859,20.986683,20.204980
consistent_hash,500,500,0,0,0.0,53.051641,35.172224,164.614677,173.263311,0.825392,0.815709,1.285842,1.584450,2.983255,2211.439296,21.113874,20.269852
prefix,500,500,0,0,0.0,49.779084,44.368267,57.368040,188.580513,0.886559,0.860120,1.425228,1.854833,2.984130,2221.099820,21.003813,20.299573


## Assembled chat completions (pod + cached_tokens)

Source: `results/<run-id>/assembled_chat_completions.json` written by `run_benchmark` after GuideLLM
(Proxy tees every streamed sim response). Skips run-ids that have no assembled file yet.

`cached_tokens` = how many tokens of **that** prompt were already in the sim KV (block-sized),
not a per-sim counter that grows 0→16→32…

In [2]:
def _cached_tokens(completion: dict) -> float:
    usage = (completion.get("chat_completion") or {}).get("usage") or {}
    detail = (
        usage.get("prompt_tokens_detail")
        or usage.get("prompt_tokens_details")
        or {}
    )
    value = detail.get("cached_tokens")
    return float(value) if value is not None else float("nan")


def load_assembled(router: str) -> pd.DataFrame | None:
    """One row per captured completion, or None if the file is missing."""
    path = RESULTS_DIR / router / ASSEMBLED_NAME
    if not path.is_file():
        warnings.warn(f"skip assembled for {router}: missing {path}")
        return None
    data = json.loads(path.read_text(encoding="utf-8"))
    rows = []
    for item in data.get("completions") or []:
        rows.append(
            {
                "router": router,
                "pod": item.get("pod") or "",
                "backend": item.get("backend") or "",
                "cached_tokens": _cached_tokens(item),
                "prompt_tokens": (
                    (item.get("chat_completion") or {}).get("usage") or {}
                ).get("prompt_tokens"),
            }
        )
    if not rows:
        warnings.warn(f"assembled file empty for {router}: {path}")
        return None
    return pd.DataFrame(rows)


assembled_frames = [df for df in (load_assembled(r) for r in ROUTERS) if df is not None]
if not assembled_frames:
    print(
        f"No {ASSEMBLED_NAME} found under {RESULTS_DIR} for {ROUTERS}. "
        "Re-run benchmarks with the capture-enabled stack."
    )
else:
    assembled = pd.concat(assembled_frames, ignore_index=True)

    cache_summary_rows = []
    for router, group in assembled.groupby("router"):
        cached = group["cached_tokens"]
        n = len(group)
        cache_summary_rows.append(
            {
                "router": router,
                "n_completions": n,
                "n_pods": group["pod"].replace("", pd.NA).nunique(dropna=True),
                "cached_mean": cached.mean(),
                "cached_median": cached.median(),
                "frac_cached_0": (cached == 0).mean(),
                "frac_cached_16": (cached == 16).mean(),
                "frac_cached_ge_600": (cached >= 600).mean(),
            }
        )
    cache_summary = pd.DataFrame(cache_summary_rows).set_index("router")
    display(cache_summary)

    pod_counts = (
        assembled.groupby(["router", "pod"])
        .size()
        .unstack(fill_value=0)
        .sort_index(axis=1)
    )
    display(pod_counts)

    pod_long = (
        pod_counts.reset_index()
        .melt(id_vars="router", var_name="pod", value_name="count")
    )
    fig_pods = px.bar(
        pod_long,
        x="pod",
        y="count",
        color="router",
        barmode="group",
        title="Requests per sim (pod)",
        labels={"count": "requests", "pod": "sim"},
    )
    fig_pods.update_layout(height=450, width=900)
    fig_pods.show()

    cache_vc = (
        assembled.assign(
            cached_tokens=assembled["cached_tokens"].fillna(-1).astype(int)
        )
        .groupby(["router", "cached_tokens"], as_index=False)
        .size()
        .rename(columns={"size": "count"})
    )
    fig_cache = px.bar(
        cache_vc,
        x="cached_tokens",
        y="count",
        color="router",
        barmode="group",
        title="cached_tokens value counts",
        labels={"cached_tokens": "cached_tokens", "count": "requests"},
    )
    fig_cache.update_layout(height=450, width=900)
    fig_cache.update_xaxes(type="category")
    fig_cache.show()

    if SAVE_PLOTS:
        PLOTS_DIR.mkdir(parents=True, exist_ok=True)
        fig_pods.write_html(PLOTS_DIR / "assembled_pods.html")
        fig_cache.write_html(PLOTS_DIR / "assembled_cached_tokens.html")

,n_completions,n_pods,cached_mean,cached_median,frac_cached_0,frac_cached_16,frac_cached_ge_600
router,,,,,,,
consistent_hash,500,8,588.512000,672.0,0.016000,0.112,0.872000
p2c,500,8,588.640000,672.0,0.016000,0.112,0.872000
prefix,493,8,662.750507,672.0,0.016227,0.000,0.983773
round_robin,500,8,588.544000,672.0,0.016000,0.112,0.872000


pod,sim-0,sim-1,sim-2,sim-3,sim-4,sim-5,sim-6,sim-7
router,,,,,,,,
consistent_hash,44,62,61,76,53,81,66,57
p2c,50,74,60,64,65,49,69,69
prefix,62,63,62,61,61,63,58,63
round_robin,62,62,62,61,64,65,62,62


## Metrics inventory (consultant follow-up)

Four layers — do not mix them:

1. **GuideLLM (client)** — TTFT / E2E / RPS / errors in `results/*/guidellm.json`. Sees the system from outside; does **not** record which sim handled a request or sim `cached_tokens` (`null` in GuideLLM JSON).
2. **Assembled completions** — `assembled_chat_completions.json` from the Proxy tee during the same run: `pod` (`x-inference-pod`) + full/reassembled `chat_completion` including `usage…cached_tokens`. This is the file that proves cache affinity for the thesis, not Prometheus.
3. **Ray Serve request router** — chooses among **Proxy** replicas using Serve-internal signals (e.g. queue length for P2C; prompt prefix for PrefixCache). Not the same as scraping the mock.
4. **llm-d-inference-sim `/metrics`** — vLLM-compatible Prometheus series on each sim. Useful inventory; our Prometheus job currently scrapes only `ray-serve:8080`, not the sims, and GuideLLM does not use these series.

Goal here: clarify meanings and gaps — **not** pick new experiment configs yet.

### GuideLLM metric glossary

**Primary** (router comparison story — already in the table/plots above):

| Metric | Meaning |
|--------|---------|
| `ttft_*` | Time to first token (ms) — how long until the client sees the first token |
| `e2e_*` | End-to-end request latency (s) |
| `throughput_rps` | Completed requests per second |
| `error_rate` / error counts | Failed or incomplete requests |

**Secondary** (useful context, not the main comparison):

| Metric | Meaning |
|--------|---------|
| `tpot_mean_ms` | Time per output token |
| `itl_mean_ms` | Inter-token latency |
| `throughput_tokens_s` | Output tokens per second |

These are all **client-side**. For replica balance and KV hit rate, use the **assembled** section above.

### Live probe: what does one mock expose?

Requires compose sims up (`sim-0` published on host `localhost:8001`). Soft-fails if unreachable.

In [3]:
import re
from urllib.error import URLError
from urllib.request import urlopen

SIM_METRICS_URL = "http://localhost:8001/metrics"
KEEP = re.compile(r"(vllm_|kv_|gpu_|prefix)", re.I)

try:
    with urlopen(SIM_METRICS_URL, timeout=5) as resp:
        text = resp.read().decode("utf-8", errors="replace")
except (URLError, TimeoutError, OSError) as exc:
    print(f"Sim /metrics unreachable at {SIM_METRICS_URL}: {exc}")
    print("Start compose (sim-0 on host port 8001) and re-run this cell.")
else:
    names = sorted(
        {
            line.split("{", 1)[0].split()[0]
            for line in text.splitlines()
            if line and not line.startswith("#") and KEEP.search(line)
        }
    )
    sample = [
        line
        for line in text.splitlines()
        if line and not line.startswith("#") and KEEP.search(line)
    ][:20]
    print(f"Matched metric names ({len(names)}):")
    for name in names:
        print(f"  {name}")
    print("\nSample lines:")
    for line in sample:
        print(line)

Matched metric names (4):
  vllm:cache_config_info
  vllm:kv_cache_usage_perc
  vllm:prefix_cache_hits
  vllm:prefix_cache_queries

Sample lines:
vllm:cache_config_info{block_size="16",num_gpu_blocks="1024"} 1
vllm:kv_cache_usage_perc{model_name="Qwen/Qwen2.5-1.5B-Instruct"} 0
vllm:prefix_cache_hits{model_name="Qwen/Qwen2.5-1.5B-Instruct"} 41664
vllm:prefix_cache_queries{model_name="Qwen/Qwen2.5-1.5B-Instruct"} 44269


### Gap table: router need vs what this stack provides

What each routing approach needs vs what GuideLLM / Serve / assembled capture / mock `/metrics` give us today.

In [4]:
gap = pd.DataFrame(
    [
        {
            "router": "PowerOfTwoChoices (default)",
            "needs": "Serve replica queue length / load among Proxy replicas",
            "guideLLM": "no (client latency only)",
            "assembled": "pod counts only (no routing signal)",
            "serve": "yes (internal)",
            "sim_/metrics": "no (not used by this router)",
        },
        {
            "router": "RoundRobin (experimental)",
            "needs": "Replica list only — no load signal",
            "guideLLM": "no",
            "assembled": "pod counts (should be flat)",
            "serve": "yes (replica set)",
            "sim_/metrics": "no",
        },
        {
            "router": "ConsistentHashing (experimental)",
            "needs": "Stable session/key for affinity",
            "guideLLM": "no",
            "assembled": "pod skew if session sticky",
            "serve": "yes (hash key from request)",
            "sim_/metrics": "no",
        },
        {
            "router": "PrefixCacheAffinityRouter",
            "needs": "Prompt/prefix; cache locality on pinned Proxy→sim",
            "guideLLM": "TTFT/E2E only (cached_tokens null)",
            "assembled": "yes — pod + usage.cached_tokens",
            "serve": "yes (native PrefixCache on Ingress→Proxy)",
            "sim_/metrics": "not scraped into router; optional inventory",
        },
        {
            "router": "Hypothetical load-/KV-aware",
            "needs": "Per-sim queue, KV/prefix hit, GPU util",
            "guideLLM": "no",
            "assembled": "post-hoc cache evidence only",
            "serve": "not wired to sims",
            "sim_/metrics": "candidate source; not scraped today",
        },
    ]
)
gap

,router,needs,guideLLM,assembled,serve,sim_/metrics
0,PowerOfTwoChoices (default),Serve replica queue length / load among Proxy ...,no (client latency only),pod counts only (no routing signal),yes (internal),no (not used by this router)
1,RoundRobin (experimental),Replica list only — no load signal,no,pod counts (should be flat),yes (replica set),no
2,ConsistentHashing (experimental),Stable session/key for affinity,no,pod skew if session sticky,yes (hash key from request),no
3,PrefixCacheAffinityRouter,Prompt/prefix; cache locality on pinned Proxy→sim,TTFT/E2E only (cached_tokens null),yes — pod + usage.cached_tokens,yes (native PrefixCache on Ingress→Proxy),not scraped into router; optional inventory
4,Hypothetical load-/KV-aware,"Per-sim queue, KV/prefix hit, GPU util",no,post-hoc cache evidence only,not wired to sims,candidate source; not scraped today


### Takeaway

GuideLLM answers **client QoS**. Serve routers use **Serve-side** signals. For **which sim** and **`cached_tokens`**, use `assembled_chat_completions.json` (Proxy capture) — not Prometheus (currently Ray-only) and not the GuideLLM JSON fields.

Do **not** pick new experiment configs from this inventory section alone.